### Imports

In [1]:
from testgen.utils import *
import pandas as pd
import json
import os
from tqdm.notebook import tqdm
from pathlib import Path
from datetime import datetime as dt
from dotenv import load_dotenv

### EDA

In [2]:
base_path = Path().cwd()
data_path = base_path / "data"
load_dotenv()

True

In [3]:
# read data and rename columns
sensor_df = pd.read_excel(data_path / "sensor_requirements.xlsx")
sensor_df_examples = pd.read_excel(data_path / "sensor_examples.xlsx")

actuator_df = pd.read_excel(data_path / "actuator_requirements.xlsx")
actuator_df_examples = pd.read_excel(data_path / "actuator_examples.xlsx")

In [4]:
sensor_df.columns = list(map(lambda x: x.lower().strip(), sensor_df.columns))
sensor_df_examples.columns = list(map(lambda x: x.lower().strip(), sensor_df_examples.columns))
actuator_df.columns = list(map(lambda x: x.lower().strip(), actuator_df.columns))
actuator_df_examples.columns = list(map(lambda x: x.lower().strip(), actuator_df_examples.columns))

In [5]:
sensor_df["target_actuator"] = 0
sensor_df_examples["target_actuator"] = 0
actuator_df["target_actuator"] = 1
actuator_df_examples["target_actuator"] = 1

In [6]:
sensor_df = sensor_df[["requirement", "target_actuator"]]
sensor_df_examples = sensor_df_examples[["requirement", "target_actuator"]]
actuator_df = actuator_df[["requirement", "target_actuator"]]
actuator_df_examples = actuator_df_examples[["requirement", "target_actuator"]]

# Format Response

In [7]:
from pydantic import BaseModel, Field

class TargetActuator(BaseModel):
    target_actuator: int = Field(description="Target actuator (0 for sensor, 1 for actuator)")

# Formulate Examples

In [8]:
N_EXAMPLES = 3

example_template = """<Example {i}>
text: {text}
target_actuator: {target_actuator}
</Example {i}>"""

examples = sensor_df_examples.sample(N_EXAMPLES).copy().to_dict(orient="records")
examples.extend(
    actuator_df_examples.sample(N_EXAMPLES).copy().to_dict(orient="records")
)


examples_text = [
    example_template.format(i=i, text=e["requirement"], target_actuator=e["target_actuator"])
    for i, e in enumerate(examples, start=1)
]
examples_text = "\n".join(examples_text)

# LLM

In [9]:
from testgen.prompts.SensorActuator import SensorActuator

In [10]:
llm_models = {
    "azure": ["gpt-4o-mini", "gpt-4o"],
    # "groq": [
        # "llama3-8b-8192", #"llama-3.2-3b-preview",
        # "llama-3.1-8b-instant",
        # "llama3-70b-8192",
        #"mixtral-8x7b-32768",
        # "llama-3.1-8b-instant"
    # ],
}

endpoint_attrs = {
    "azure": {
        "api_key": os.getenv("AZURE_OPENAI_API_KEY"),
        "api_version": os.getenv("AZURE_API_VERSION"),
        "azure_endpoint": os.getenv("AZURE_OPENAI_ENDPOINT"),
    },
    "groq": {
        "api_key": os.getenv("GROQ_API_KEY"),
    },
}

# Utils for all Requirements

In [11]:
# # combine both dataframes and shuffle
# df = (
#     pd.concat([sensor_df, actuator_df], ignore_index=True)
#     .sample(frac=1)
#     .reset_index(drop=True)
# )

df = pd.read_excel(data_path / "sensor_reqs_rewritten.xlsx")
df = df[["requirement"]]
df["target_actuator"] = 0

In [17]:
def run_all_requirements(endpoint_name, client, df, model_name):
    results = []
    responses = []

    for instance in tqdm(df.iterrows(), total=df.shape[0]):

        # system prompt
        messages = [
            {
                "role": "system",
                "content": SensorActuator.format(
                    examples=examples_text,
                    requirement=instance[1].iloc[0],
                ),
            }
        ]

        # run LLM
        result = {
            "requirement": instance[1].iloc[0],
            "target_actuator": instance[1].iloc[1],
            "model": model_name,
        }
        start_time = time.perf_counter()
        if endpoint_name == "azure":
            response = client.beta.chat.completions.parse(
                model=model_name,
                messages=messages,
                temperature=0.2,
                max_tokens=256,
                response_format=TargetActuator,
            )
        elif endpoint_name == "groq":

            # add format response to prompt
            messages[0]["content"] = messages[0]["content"].replace(
                "</Solution Plan>",
                f"""5. Format the response in JSON format.
</Solution Plan>

<Output Format>
The JSON object must use the schema: {json.dumps(TargetActuator.model_json_schema(), indent=2)}
</Output Format>""",
            )

            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=0.2,
                max_tokens=256,
                response_format={"type": "json_object"},
            )

        response_time = round(time.perf_counter() - start_time, 6)
        result["ai_response"] = response.choices[0].message.content
        messages.append(
            {
                "role": "assistant",
                "content": result["ai_response"],
            }
        )

        result_json = json.loads(result["ai_response"])
        result["ai_answer"] = result_json["target_actuator"]
        result["response_time"] = response_time
        result["accuracy"] = instance[1].iloc[1] == result_json["target_actuator"]

        response_usage = response.usage.to_dict()
        if endpoint_name == "azure":
            response_usage.pop("completion_tokens_details")
            response_usage.pop("prompt_tokens_details")

        result.update(response_usage)

        results.append(result)
        responses.append(response)

    return results, responses

In [18]:
def calc_stats(results):
    accuracy = 0
    total_tokens = 0
    total_completion_tokens = 0
    total_time = 0

    for r in results:
        accuracy += r["accuracy"]
        total_tokens += r["total_tokens"]
        total_completion_tokens += r["completion_tokens"]
        total_time += r["response_time"]

    number_of_reqs = len(results)
    accuracy /= len(results)
    avg_time_per_req = round(total_time / len(results), 6)
    avg_token_per_req = total_tokens / len(results)
    avg_completion_token_per_req = total_completion_tokens / len(results)

    return (
        number_of_reqs,
        accuracy,
        avg_time_per_req,
        avg_token_per_req,
        avg_completion_token_per_req,
        total_tokens,
        total_completion_tokens,
        total_time,
    )

In [19]:
def save_responses(**kwargs):
    # save results
    time = dt.now()

    results_path = (
        "results/sensor-actuator_new_single_{model}_n-{examples}_acc-{accuracy}_{time}.json"
    )
    results_path = results_path.format(
        model=kwargs["model_name"],
        examples=N_EXAMPLES,
        time=time.strftime("%m.%d.%Y-%H:%M:%S"),
        accuracy=round(kwargs["accuracy"], 3),
    )

    results_file = base_path / results_path
    results_file.parent.mkdir(exist_ok=True)
    results_file.touch()

    with results_file.open("w") as f:
        json.dump(
            {
                "accuracy": kwargs["accuracy"],
                "number_of_reqs": kwargs["number_of_reqs"],
                "total_tokens": kwargs["total_tokens"],
                "total_completion_tokens": kwargs["total_completion_tokens"],
                "avg_token_per_req": kwargs["avg_token_per_req"],
                "avg_completion_token_per_req": kwargs["avg_completion_token_per_req"],
                "avg_time_per_req": kwargs["avg_time_per_req"],
                "examples": examples,
                "responses": kwargs["results"],
            },
            f,
            indent=4,
        )

    return results_file

# Run against all models

In [20]:
for endpoint_name in llm_models.keys():
    for model_name in llm_models[endpoint_name]:

        print(f"Running {model_name} on {endpoint_name}...")

        client = llm_client(endpoint_name, **endpoint_attrs[endpoint_name])

        results, responses = run_all_requirements(endpoint_name, client, df, model_name)

        (
            number_of_reqs,
            accuracy,
            avg_time_per_req,
            avg_token_per_req,
            avg_completion_token_per_req,
            total_tokens,
            total_completion_tokens,
            total_time,
        ) = calc_stats(results)

        results_file = save_responses(
            model_name=model_name,
            accuracy=accuracy,
            number_of_reqs=number_of_reqs,
            total_tokens=total_tokens,
            total_completion_tokens=total_completion_tokens,
            avg_token_per_req=avg_token_per_req,
            avg_completion_token_per_req=avg_completion_token_per_req,
            avg_time_per_req=avg_time_per_req,
            results=results,
        )

        print(f"Done")
        print(f"|+=+=+| " * 10)

Running gpt-4o-mini on azure...


  0%|          | 0/40 [00:00<?, ?it/s]

Done
|+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| 
Running gpt-4o on azure...


  0%|          | 0/40 [00:00<?, ?it/s]

Done
|+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| |+=+=+| 


In [107]:
results[0]

{'requirement': 'The vehicle control system must have the capability to override accelerator pedal inputs in the event of an identified fault that poses a risk to vehicle safety',
 'target_actuator': 0,
 'model': 'llama-3.1-8b-instant',
 'ai_response': '{\n   "target_actuator": 1\n}',
 'ai_answer': 1,
 'response_time': 0.471055,
 'accuracy': False,
 'completion_tokens': 11,
 'prompt_tokens': 795,
 'total_tokens': 806,
 'completion_time': 0.014666667,
 'prompt_time': 0.074120341,
 'queue_time': 0.088551565,
 'total_time': 0.088787008}